In [1]:
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
import random

In [2]:
from datasets import load_dataset
print("Hugging Face Datasets imported successfully")

Hugging Face Datasets imported successfully


In [3]:
from datasets import get_dataset_config_names, load_dataset_builder

dataset_name = 'liaad/PtBrVId'

configs = get_dataset_config_names(dataset_name)
print("available configuration")
for config in configs: 
    print("-", config)

available configuration
- journalistic
- legal
- literature
- politics
- social_media
- web


In [4]:
from datasets import load_dataset
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score

In [7]:
from itertools import islice

stream = load_dataset(
    dataset_name,
    name="journalistic",
    split="train",
    streaming=True
)

for position in [0, 100_000, 500_000, 1_000_000, 1_500_000]:
    stream = load_dataset(
        dataset_name,
        name="journalistic",
        split="train",
        streaming=True
    )

    records = list(islice(stream, position, position + 100))

    print(
        f"Position {position}:",
        pd.Series([x["label"] for x in records]).value_counts().to_dict()
    )

Position 0: {0: 100}
Position 100000: {0: 100}
Position 500000: {0: 100}
Position 1000000: {0: 100}
Position 1500000: {1: 100}


In [25]:
from itertools import islice

stream = load_dataset(
    dataset_name,
    name="journalistic",
    split="train",
    streaming=True
)

for position in range(1_000_000, 1_600_000, 50_000):
    stream = load_dataset(
        dataset_name,
        name="journalistic",
        split="train",
        streaming=True
    )

    records = list(islice(stream, position, position + 100))

    labels = pd.Series([x["label"] for x in records])

    print(position, labels.value_counts().to_dict())

1000000 {0: 100}
1050000 {0: 100}
1100000 {0: 100}
1150000 {0: 100}
1200000 {0: 100}
1250000 {0: 100}
1300000 {0: 100}
1350000 {0: 100}
1400000 {0: 100}
1450000 {1: 100}
1500000 {1: 100}
1550000 {1: 100}


In [5]:
random.seed(42)

dataset_name = "liaad/PtBrVId"
target_per_class = 10_000

reservoirs = {
    0: [],
    1: []
}

counts = {
    0: 0,
    1: 0
}

train_stream = load_dataset(
    dataset_name,
    name="journalistic",
    split="train",
    streaming=True
)

for example in train_stream:

    label = example["label"]
    counts[label] += 1

    reservoir = reservoirs[label]
    count = counts[label]

    if len(reservoir) < target_per_class:
        reservoir.append(example)

    else:
        random_index = random.randrange(count)

        if random_index < target_per_class:
            reservoir[random_index] = example

train_df = pd.DataFrame(
    reservoirs[0] + reservoirs[1]
)

train_df = train_df.sample(
    frac=1,
    random_state=42
).reset_index(drop=True)

print("Original class counts:", counts)
print("\nSample distribution:")
print(train_df["label"].value_counts())

Original class counts: {0: 1442906, 1: 333384}

Sample distribution:
label
1    10000
0    10000
Name: count, dtype: int64


In [6]:
X = train_df["text"]
y = train_df["label"]

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y,
    test_size=0.30,
    random_state=42,
    stratify=y
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.50,
    random_state=42,
    stratify=y_temp
)

print("Training:", y_train.value_counts().to_dict())
print("Validation:", y_val.value_counts().to_dict())
print("Test:", y_test.value_counts().to_dict())

Training: {0: 7000, 1: 7000}
Validation: {0: 1500, 1: 1500}
Test: {1: 1500, 0: 1500}


In [7]:
baseline_model = Pipeline([
    (
        "tfidf",
        TfidfVectorizer(
            analyzer="word",
            ngram_range=(1, 2),
            min_df=2,
            max_features=100_000,
            sublinear_tf=True
        )
    ),
    (
        "classifier",
        LogisticRegression(
            max_iter=1000,
            random_state=42
        )
    )
])

In [8]:
baseline_model.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('tfidf', ...), ('classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
,"ngram_range ngram_range: tuple (min_n, max_n), default=(1, 1)The lower and upper boundary of the range of n-values for differentn-grams to be extracted. All values of n such that min_n <= n <= max_nwill be used. For example an ``ngram_range`` of ``(1, 1)`` means onlyunigrams, ``(1, 2)`` means unigrams and bigrams, and ``(2, 2)`` meansonly bigrams.Only applies if ``analyzer`` is not callable.","(1, ...)"
,"min_df min_df: float or int, default=1When building the vocabulary ignore terms that have a documentfrequency strictly lower than the given threshold. This value is alsocalled cut-off in the literature.If float in range of [0.0, 1.0], the parameter represents a proportionof documents, integer absolute counts.This parameter is ignored if vocabulary is not None.",2
,"max_features max_features: int, default=NoneIf not None, build a vocabulary that only consider the top`max_features` ordered by term frequency across the corpus.Otherwise, all features are used.This parameter is ignored if vocabulary is not None.",100000
,"sublinear_tf sublinear_tf: bool, default=FalseApply sublinear tf scaling, i.e. replace tf with 1 + log(tf).",True
,"input input: {'filename', 'file', 'content'}, default='content'- If `'filename'`, the sequence passed as an argument to fit is expected to be a list of filenames that need reading to fetch the raw content to analyze.- If `'file'`, the sequence items must have a 'read' method (file-like object) that is called to fetch the bytes in memory.- If `'content'`, the input is expected to be a sequence of items that can be of type string or byte.",'content'


In [9]:
y_val_pred = baseline_model.predict(X_val)

print("Accuracy:", accuracy_score(y_val, y_val_pred))
print("Macro F1:", f1_score(y_val, y_val_pred, average="macro"))

print("\nClassification Report:")
print(classification_report(y_val, y_val_pred))

Accuracy: 0.941
Macro F1: 0.940999940999941

Classification Report:
              precision    recall  f1-score   support

           0       0.94      0.94      0.94      1500
           1       0.94      0.94      0.94      1500

    accuracy                           0.94      3000
   macro avg       0.94      0.94      0.94      3000
weighted avg       0.94      0.94      0.94      3000



In [10]:
vectorizer = baseline_model.named_steps["tfidf"]
classifier = baseline_model.named_steps["classifier"]

feature_names = vectorizer.get_feature_names_out()
coefficients = classifier.coef_[0]

top_positive = np.argsort(coefficients)[-30:][::-1]
top_negative = np.argsort(coefficients)[:30]

print("Features associated with Label 1:")
for i in top_positive:
    print(f"{feature_names[i]:30} {coefficients[i]:.4f}")

print("\nFeatures associated with Label 0:")
for i in top_negative:
    print(f"{feature_names[i]:30} {coefficients[i]:.4f}")

Features associated with Label 1:
folha                          6.4281
ele                            5.1981
brasil                         5.0009
us                             4.8030
sao paulo                      4.0515
disse                          3.7220
equipe                         3.3859
paulo                          3.3612
rio                            2.9759
fhc                            2.9489
reportagem                     2.8595
reportagem local               2.8266
diz                            2.8251
da reportagem                  2.8160
de sua                         2.7835
time                           2.7780
em um                          2.7049
diretor                        2.7044
voce                           2.7034
lula                           2.6805
projeto                        2.5835
brasileira                     2.5101
copa                           2.5028
disse que                      2.4542
selecao                        2.4540
fato            

In [11]:
def show_top_features(features, coefficients, n=50):
    for i in features:
        print(
            f"{feature_names[i]:30s} "
            f"{coefficients[i]:.4f}"
        )
    
print("LABEL 1")
show_top_features(
    np.argsort(coefficients)[-50:][::-1],
    coefficients
)

print("\nLABEL 0")
show_top_features(
    np.argsort(coefficients)[:50],
    coefficients
)

LABEL 1
folha                          6.4281
ele                            5.1981
brasil                         5.0009
us                             4.8030
sao paulo                      4.0515
disse                          3.7220
equipe                         3.3859
paulo                          3.3612
rio                            2.9759
fhc                            2.9489
reportagem                     2.8595
reportagem local               2.8266
diz                            2.8251
da reportagem                  2.8160
de sua                         2.7835
time                           2.7780
em um                          2.7049
diretor                        2.7044
voce                           2.7034
lula                           2.6805
projeto                        2.5835
brasileira                     2.5101
copa                           2.5028
disse que                      2.4542
selecao                        2.4540
fato                           2.4150
de s

**Character level base line**

In [12]:
char_model = Pipeline([
    ("tfidf", TfidfVectorizer(
        analyzer="char",
        ngram_range=(3, 5),
        min_df=3,
        max_features=200_000,
        sublinear_tf=True
    )),
    ("classifier", LogisticRegression(
        max_iter=1000,
        random_state=42
    ))
])

char_model.fit(X_train, y_train)

char_predictions = char_model.predict(X_val)

print("Accuracy:", accuracy_score(y_val, char_predictions))
print("Macro F1:", f1_score(
    y_val,
    char_predictions,
    average="macro"
))

print(classification_report(y_val, char_predictions))

Accuracy: 0.942
Macro F1: 0.9419906927289223
              precision    recall  f1-score   support

           0       0.95      0.93      0.94      1500
           1       0.93      0.95      0.94      1500

    accuracy                           0.94      3000
   macro avg       0.94      0.94      0.94      3000
weighted avg       0.94      0.94      0.94      3000



**Combined feature model**

In [13]:
combined_model = Pipeline([
    ("features", FeatureUnion([
        ("word", TfidfVectorizer(
            analyzer="word",
            ngram_range=(1, 2),
            min_df=2,
            max_features=100_000,
            sublinear_tf=True
        )),
        ("char", TfidfVectorizer(
            analyzer="char",
            ngram_range=(3, 5),
            min_df=3,
            max_features=200_000,
            sublinear_tf=True
        ))
    ])),
    ("classifier", LogisticRegression(
        max_iter=1000,
        random_state=42
    ))
])

combined_model.fit(X_train, y_train)

combined_predictions = combined_model.predict(X_val)

print("Accuracy:", accuracy_score(y_val, combined_predictions))
print("Macro F1:", f1_score(
    y_val,
    combined_predictions,
    average="macro"
))

print(classification_report(y_val, combined_predictions))

Accuracy: 0.9533333333333334
Macro F1: 0.9533280230995171
              precision    recall  f1-score   support

           0       0.96      0.94      0.95      1500
           1       0.94      0.96      0.95      1500

    accuracy                           0.95      3000
   macro avg       0.95      0.95      0.95      3000
weighted avg       0.95      0.95      0.95      3000



In [14]:
feature_union = combined_model.named_steps["features"]
classifier = combined_model.named_steps["classifier"]

word_vectorizer = feature_union.transformer_list[0][1]
char_vectorizer = feature_union.transformer_list[1][1]

word_features = word_vectorizer.get_feature_names_out()
char_features = char_vectorizer.get_feature_names_out()

n_word_features = len(word_features)

coefficients = classifier.coef_[0]

word_coefficients = coefficients[:n_word_features]
char_coefficients = coefficients[n_word_features:]

print("Word features:", len(word_features))
print("Character features:", len(char_features))

Word features: 100000
Character features: 194206


In [15]:
def show_top_features(features, coefficients, n=30):
    indices = np.argsort(coefficients)

    print("\nAssociated with label 0:")
    for i in indices[:n]:
        print(f"{features[i]:30s} {coefficients[i]:.4f}")

    print("\nAssociated with label 1:")
    for i in indices[-n:][::-1]:
        print(f"{features[i]:30s} {coefficients[i]:.4f}")


print("WORD FEATURES")
show_top_features(word_features, word_coefficients)

print("\nCHARACTER FEATURES")
show_top_features(char_features, char_coefficients)

WORD FEATURES

Associated with label 0:
lisboa                         -2.8930
cento                          -2.8042
por cento                      -2.7837
se                             -2.6066
contos                         -2.5069
portugal                       -2.4935
lhe                            -2.4350
equipa                         -2.2525
numa                           -2.1365
os seus                        -1.9933
que se                         -1.9663
manuel                         -1.8444
porto                          -1.8315
facto                          -1.7295
num                            -1.6737
psd                            -1.5794
da sua                         -1.5753
direccao                       -1.5122
do seu                         -1.5022
dos                            -1.4609
os                             -1.4344
de contos                      -1.4122
ps                             -1.4070
projecto                       -1.3982
as                      

Now given the fact that the model has learned from certain geographical references such as "portugal", "brasil" and "lisboa". Now we're going to test how much the model actually depends on those terms

In [16]:
import re

geographical_terms = [
    "portugal",
    "brasil",
    "lisboa",
    "porto",
    "sao paulo",
    "rio de janeiro",
]

def mask_geographical_terms(text):
    for term in geographical_terms:
        pattern = re.compile(re.escape(term), flags=re.IGNORECASE)
        text = pattern.sub(" GEOENTITY ", text)
    return text

X_train_masked = X_train.apply(mask_geographical_terms)
X_val_masked = X_val.apply(mask_geographical_terms)

In [17]:
masked_model = Pipeline([
    ("features", FeatureUnion([
        ("word", TfidfVectorizer(
            analyzer="word",
            ngram_range=(1, 2),
            min_df=2,
            max_features=100_000,
            sublinear_tf=True
        )),
        ("char", TfidfVectorizer(
            analyzer="char",
            ngram_range=(3, 5),
            min_df=3,
            max_features=200_000,
            sublinear_tf=True
        ))
    ])),
    ("classifier", LogisticRegression(
        max_iter=1000,
        random_state=42
    ))
])

masked_model.fit(X_train_masked, y_train)

masked_predictions = masked_model.predict(X_val_masked)

print("Masked accuracy:",
      accuracy_score(y_val, masked_predictions))

print("Masked macro F1:",
      f1_score(y_val, masked_predictions, average="macro"))

Masked accuracy: 0.9513333333333334
Masked macro F1: 0.951326324324036


In [18]:
import joblib
from pathlib import Path

model_dir = Path("../models")
model_dir.mkdir(exist_ok=True)

joblib.dump(
    combined_model,
    model_dir / "combined_tfidf_logistic_regression.joblib"
)

['../models/combined_tfidf_logistic_regression.joblib']